# カスタム Model Context Protocol（MCP）ツールを使用したエージェントの構築

## 概要

このノートブックでは、カスタム **Model Context Protocol（MCP）** サーバーを構築し、Google **Agent Development Kit（ADK）** と統合して AI エージェントを構築する方法を学びます。

**Model Context Protocol（MCP）** は、統一されたプロトコルを通じて AI モデルやエージェントが外部データソース、API、ツールとやり取りできるようにするオープンスタンダードです。新しいデータソースごとに個別のツール統合ロジックを作成する代わりに、MCP を使用すると、標準的なサーバー（Server-Sent Events / SSE や Standard I/O などのトランスポートメカニズムを使用）経由でツールを公開でき、互換性のある任意のエージェントから接続できるようになります。

このチュートリアルでは、以下を行います。
* `FastMCP` SDK を使用して、在庫の照会と更新を行うツールを公開する **MCP サーバー**（`mcp_server.py`）を構築します。
* `McpToolset` と `SseServerParams` を使用して、SSE 経由で MCP サーバーに接続する **ADK エージェント**（`LlmAgent`）を設定します。
* **ADK Developer UI**（`adk web`）を使用して MCP 対応エージェントを操作し、ツール呼び出しの検査、対話型ワークフローのテスト、在庫変更の検証を行います。
* さらに、在庫管理 MCP サーバーを本番用マイクロサービスとして **Google Cloud Run** にデプロイし、リモート ADK エージェントから接続します。

## 学習目標

このノートブックを完了すると、以下のことができるようになります。
* **MCP アーキテクチャの理解:** MCP がツールやデータのサービスプロバイダー（サーバー）と AI エージェントのコンシューマー（クライアント）をどのように分離するかを理解する。
* **MCP サーバーの構築:** `FastMCP`（`@mcp.tool()`）を使用して、Python 関数とスキーマを標準化された MCP ツールとして公開する。
* **ADK での MCP ツールの統合:** `McpToolset` と `SseServerParams` を使用して、ADK の `LlmAgent` を MCP サーバーに接続する。
* **ADK Developer UI の使用:** `adk web` を実行して、MCP を統合したエージェントを検査し、対話形式でテストする。
* **Cloud Run への MCP サーバーのデプロイ:** MCP サーバーをパッケージ化して Google Cloud Run にデプロイし、リモート ADK エージェントをその `/sse` HTTPS エンドポイントに接続する。

## セットアップとインポート

まずは、ADK、MCP、非同期実行に必要な Python ライブラリをインポートします。

In [ ]:
import asyncio
import csv
import importlib
import logging
import os
import socket
import subprocess
import sys
import time
import warnings

import pandas as pd
import google.auth
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.genai import types

if "." not in sys.path:
    sys.path.insert(0, ".")

# ノートブックの出力をクリーンに保つため、警告と opentelemetry のログを無視
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry.trace").setLevel(logging.CRITICAL)

In [ ]:
_, PROJECT_ID = google.auth.default()
GEMINI_LOCATION = "global"
CLOUD_RUN_LOCATION = "us-central1"
MODEL = "gemini-3.5-flash"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
os.environ["CLOUD_RUN_LOCATION"] = CLOUD_RUN_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"


## 在庫 MCP エージェントの構築

このセクションでは、ローカル MCP ツールエージェントの完全なパイプラインを構築します。具体的には、在庫データセットの作成、読み取りおよび変更ツールを備えた MCP サーバーの構築、ローカル SSE サーバーの起動、ADK エージェントの定義、そして Developer UI（`adk web`）を使用した対話型テストを行います。

In [ ]:
!mkdir -p ./custom_mcp_agent/agent


In [ ]:
%%writefile ./custom_mcp_agent/agent/__init__.py
# pylint: skip-file
from . import agent


### エージェント環境設定（`.env`）の作成

エージェントモジュール `./custom_mcp_agent` はデフォルトの `adk_agents` ディレクトリの外にあるため、`./custom_mcp_agent` 内に `.env` ファイルを作成し、`adk web` の実行時に ADK が Vertex AI の環境設定を自動的に読み込むようにします。

In [ ]:
%%writefile ./custom_mcp_agent/.env
GOOGLE_CLOUD_LOCATION=global
GOOGLE_GENAI_USE_VERTEXAI=TRUE


### サンプル在庫データ（`sku_data.csv`）の作成

この例では、MCP サーバーが CSV ファイル（`sku_data.csv`）に保存された在庫データベースを管理します。
各レコードには、製品の SKU、名前、カテゴリ、手持ち数量、単価、在庫ステータスなどの詳細が含まれています。

モジュールディレクトリ `./custom_mcp_agent` を作成し、そこにサンプルの CSV ファイルを書き込みます。

In [ ]:
%%writefile ./custom_mcp_agent/sku_data.csv
SKU,ProductName,Category,SupplierID,SupplierName,QuantityOnHand,ReorderLevel,UnitPrice,UnitCost,LastStockInDate,LastStockOutDate,Location,Status,Notes
SKU001,"Laptop Pro 15""",Electronics,SUP001,TechGlobal,60,5,1200.00,950.00,2023-04-10,2023-05-15,Warehouse A - Shelf 1,Low Stock,High-performance model
SKU002,Wireless Mouse,Accessories,SUP002,OfficeGear,70,20,25.00,15.00,2023-03-20,2023-05-10,Warehouse B - Bin 3,Low Stock,Ergonomic design
SKU003,Mechanical Keyboard,Accessories,SUP001,TechGlobal,40,10,75.00,50.00,2023-04-01,2023-05-05,Warehouse A - Shelf 2,In Stock,RGB Backlit
SKU004,"27"" 4K Monitor",Electronics,SUP003,DisplayInc,25,5,300.00,220.00,2023-02-15,2023-04-28,Warehouse C - Area 1,In Stock,
SKU005,HD Webcam,Accessories,SUP002,OfficeGear,60,15,50.00,30.00,2023-03-05,2023-05-12,Warehouse B - Bin 5,In Stock,Includes microphone
SKU006,USB-C Hub,Accessories,SUP001,TechGlobal,30,10,35.00,20.00,2023-04-18,2023-05-01,Warehouse A - Shelf 3,Low Stock,7-in-1 adapter


### 在庫 MCP サーバー（`mcp_server.py`）の構築

次に、`mcp.server.fastmcp` パッケージの `FastMCP` SDK を使用して MCP サーバーを構築します。
実装を以下の 3 つのセクションに分けて進めます。
1. **FastMCP サーバーの初期化**: サーバーインスタンスを作成し、ホストとポートを指定します。
2. **在庫照会ツール（`list_skus`）**: CSV ファイルから SKU を検査する読み取り専用ツールを公開します。
3. **在庫変更ツール（`update_sku_qty`）とエントリポイント**: 製品の数量を更新する状態変更ツールを公開し、Server-Sent Events（SSE）サーバーを起動します。

#### FastMCP サーバーの初期化

まず必要なライブラリ（`asyncio`、`csv`、`os`、`FastMCP`、`Field`）をインポートし、`mcp.server.fastmcp` SDK から `FastMCP` サーバーインスタンスを初期化します。

* **`FastMCP(...)`**: AI クライアントがこの MCP サーバーの全体的な目的を理解できるように、わかりやすい `name` と詳細な自然言語による `instructions`（指示）を指定してサーバーを初期化します。
* **ネットワーク構成**: デフォルトでホスト `0.0.0.0` とポート `4200`（または Cloud Run にデプロイされた場合は `PORT` 環境変数で指定されたポート）でリッスンするようにサーバーを設定します。

In [ ]:
%%writefile ./custom_mcp_agent/mcp_server.py
"""在庫管理用の MCP サーバー。"""

import csv
import os

from mcp.server.fastmcp import FastMCP
from pydantic import Field

# このスクリプトに対する相対パスで CSV ファイルへのパスを定義
CSV_FILE_PATH = os.path.join(os.path.dirname(__file__), "sku_data.csv")

mcp = FastMCP(
    name="Inventory MCP Server",
    instructions="""
    This Server provides inventory related data and helps in
    updating the quantity of a specific SKU.

    Call list_skus() to get the list of all the SKUs or any details related to
    the items/SKUs.
    Call update_sku_qty(sku_id, quantity, sign) to update the quantity of a specific SKU.
    """,
    host="0.0.0.0",
    port=int(os.environ.get("PORT", 4200)),
)

#### 在庫照会ツール（`list_skus`）

次に、エージェントが在庫アイテムを照会できるようにする最初の MCP ツール `list_skus` を定義します。

* **`@mcp.tool()` デコレータ**: Python 関数を FastMCP サーバー上の MCP ツールとして登録します。
* **Pydantic スキーマアノテーション**: パラメータの説明に Pydantic の `Field("*", description="Name of the SKU")` を使用します。MCP サーバーはこれを自動的に JSON Schema に変換するため、接続するエージェントはツールが受け取る引数を把握できます。
* **Docstring による指示**: 関数を *いつ* 呼び出し、*何* を返すかを説明する詳細な docstring に注目してください。LLM は MCP を介してこの docstring を読み取り、いつツールを呼び出すかを判断します。
* **CSV 読み取りロジック**: `csv.DictReader` を使用して `sku_data.csv` を開き、すべてのレコードを辞書のリストとして返します。

In [ ]:
%%writefile -a ./custom_mcp_agent/mcp_server.py


@mcp.tool()
def list_skus(sku_name: str = Field("*", description="Name of the SKU")):
    """CSV ファイルから SKU データを読み取り、辞書のリストを返します。

    利用可能なアイテムについて尋ねられた場合や、アイテム / SKU に関する問い合わせがあった場合は、
    この関数を呼び出し、ID、名前、価格、手持ち数量などのわかりやすい情報のみを返します。
    """
    if not os.path.exists(CSV_FILE_PATH):
        return {"error": "SKU data file not found."}

    skus = []
    try:
        with open(CSV_FILE_PATH, newline="", encoding="utf-8") as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                if (
                    sku_name == "*"
                    or sku_name.lower()
                    in row.get("ProductName", "").lower()
                ):
                    skus.append(row)
        return {"skus": skus}
    except (OSError, csv.Error) as e:
        return {"error": f"Failed to read SKU data: {str(e)}"}

#### 在庫変更ツール（`update_sku_qty`）とサーバーエントリポイント

最後に、エージェントが在庫数を変更し、在庫ステータスを自動更新できるようにする 2 つ目の MCP ツール `update_sku_qty` を定義します。また、サーバーのエントリポイントも追加します。

* **ツールによる状態の変更**: このツールは `sku_id`、`quantity`、および `sign`（追加の場合は `1`、削減の場合は `-1`）を受け取ります。`sku_data.csv` を読み込み、一致する SKU の `QuantityOnHand` を更新し、`ReorderLevel` に基づいて `Status` を `'Low Stock'` または `'In Stock'` に自動的に再計算します。
* **スキーマと検証**: Pydantic の `Field(..., ge=0)` により、`quantity` が負でない整数であることを保証します。
* **サーバーエントリポイント**: `if __name__ == "__main__":` の下で `mcp.run(transport="sse")` を呼び出し、**Server-Sent Events（SSE）** トランスポートプロトコルを使用してサーバーを起動します。

In [ ]:
%%writefile -a ./custom_mcp_agent/mcp_server.py


@mcp.tool()
def update_sku_qty(
    sku_id: str = Field(
        ..., description="The SKU ID of the product to update."
    ),
    quantity: int = Field(
        ...,
        description="The quantity to be added or removed from the SKU.",
        ge=0,
    ),
    sign: int = Field(1, description="1 for adding and -1 for removing."),
):
    """CSV ファイル内の特定の SKU の数量を更新します。

    注文の確定、返品、キャンセルを求められた場合は、この関数を呼び出します。

    Args:
        sku_id (str): 更新する製品の SKU ID。
        quantity (int): SKU に追加または削減する数量。
        sign (int): 追加の場合は 1、削減の場合は -1。

    Returns:
        dict: 成功または失敗を示すメッセージ。
    """
    if not os.path.exists(CSV_FILE_PATH):
        return {"error": "SKU data file not found."}

    if not isinstance(quantity, int):
        return {"error": "Invalid quantity. Must be a non-negative integer."}

    rows = []
    updated = False
    updated_qty = 0
    fieldnames = []

    try:
        with open(CSV_FILE_PATH, newline="", encoding="utf-8") as csvfile:
            reader = csv.DictReader(csvfile)
            fieldnames = reader.fieldnames
            if not fieldnames:  # 空または不正な形式の CSV を処理
                return {"error": "CSV file is empty or has no header."}

            quantity *= sign

            for row in reader:
                if row.get("SKU") == sku_id:
                    row["QuantityOnHand"] = int(row["QuantityOnHand"]) + int(
                        quantity
                    )
                    # 新しい数量と ReorderLevel に基づいて
                    # "Status" を更新する可能性あり
                    if "ReorderLevel" in row and quantity <= int(
                        row.get("ReorderLevel", 0)
                    ):
                        row["Status"] = "Low Stock"
                    elif "ReorderLevel" in row and quantity > int(
                        row.get("ReorderLevel", 0)
                    ):
                        row["Status"] = "In Stock"
                    updated = True
                    updated_qty = row["QuantityOnHand"]
                rows.append(row)

        if not updated:
            return {"error": f"SKU ID '{sku_id}' not found."}

        with open(
            CSV_FILE_PATH, mode="w", newline="", encoding="utf-8"
        ) as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        return {
            "message": f"Quantity for SKU '{sku_id}' updated to {updated_qty}."
        }
    except (OSError, csv.Error, ValueError, KeyError) as e:
        return {"error": f"Failed to update SKU quantity: {str(e)}"}


if __name__ == "__main__":
    mcp.run(transport="sse")

### ローカルでの在庫 MCP サーバーの起動

`mcp_server.py` をバックグラウンドのサブプロセスとして起動し、ローカルの ADK エージェントがポート `4200` の `/sse` エンドポイントに接続できるようにします。先に進む前に、サーバーが正常にバインドされていることを確認します。

In [ ]:
# MCP サーバーをバックグラウンドのサブプロセスとして起動
print("Starting Inventory MCP Server on port 4200...")
server_process = subprocess.Popen(
    [sys.executable, "-m", "custom_mcp_agent.mcp_server"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# サーバーがポート 4200 にバインドされるまで最大 15 秒間待機
for _ in range(30):
    try:
        with socket.create_connection(("0.0.0.0", 4200), timeout=1):
            print("SUCCESS: MCP Server is running and listening on http://0.0.0.0:4200/sse")
            break
    except OSError:
        time.sleep(0.5)
else:
    print("ERROR: Failed to start MCP Server on port 4200.")

### `McpToolset` を使用した ADK エージェントの定義（`agent.py`）

ADK では、エージェントの `tools` リストに `McpToolset` を追加することで MCP サーバーに接続します。
* `McpToolset` は、MCP サーバーへの接続、利用可能なツールの検出、Gemini の関数呼び出し（Function Calling）宣言と MCP ツール呼び出しの間の変換を処理します。
* ここでは、`McpToolset` を `SseServerParams(url="http://0.0.0.0:4200/sse")` で設定し、Server-Sent Events（SSE）経由でローカルの在庫 MCP サーバーに接続します。

In [ ]:
%%writefile ./custom_mcp_agent/agent/agent.py
"""MCP Toolset を使用して在庫管理ツールを作成する例。"""

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

MODEL = "gemini-3.5-flash"

# TODO: 重要! 以下のパスをリモート MCP サーバーのパスに変更してください
MCP_SERVER_URL = "http://0.0.0.0:4200/sse"

root_agent = LlmAgent(
    model=Gemini(
        model=MODEL,
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    name="inventory_assistant",
    description="You are a specialized assistant for inventory management.",
    instruction=(
        "Help user get answer to their queries about inventory and update "
        "or process the items."
    ),
    tools=[
        McpToolset(
            connection_params=SseServerParams(url=MCP_SERVER_URL),
        )
    ],
)

### ADK Developer UI の起動

`adk web` コマンドを使用して ADK Developer UI を起動し、MCP を統合したエージェントと対話できます。

下のセルを実行し、出力された URL を開きます（Web サーバーが起動するまでに数秒かかる場合があります）。Web UI が開いたら、上部のエージェントドロップダウンから **`agent`** を選択し、以下のサンプル質問を送信してみてください。

* *"Can you list all available SKUs along with their product names, cost, and quantities on hand?"*（利用可能なすべての SKU について、製品名、コスト、手持ち数量を一覧表示できますか？）（`list_skus` のテスト）
* *"What is the current stock level and status of SKU001 (Laptop Pro 15)?"*（SKU001（Laptop Pro 15）の現在の在庫レベルとステータスはどうなっていますか？）（`list_skus` のテスト）
* *"We just received a new shipment of 15 units for SKU001. Please add 15 to the quantity of SKU001."*（SKU001 が新たに 15 台入荷しました。SKU001 の数量に 15 を追加してください。）（`update_sku_qty` のテスト）
* *"Which products are currently marked as Low Stock?"*（現在「Low Stock」とマークされている製品はどれですか？）（`list_skus` のフィルタリングのテスト）

**注**: ターミナルから `adk web` を実行することもできます。その場合は、事前に仮想環境がアクティベートされていることを確認してください。
```
# asl-ml-immersion ディレクトリ内
source ./asl_genai/.venv/bin/activate
adk web ./asl_genai/notebooks/building_agents/solutions/custom_mcp_agent
```

In [ ]:
# Cloud Workstations の場合
!adk web custom_mcp_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev" 

**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web custom_mcp_agent --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


### CSV ファイルでのデータの永続化の確認

Developer UI でエージェントをテストし、カーネルを中断（インターラプト）した後、下のセルを実行して、基盤となるファイルで `SKU001` の数量が更新されたことを確認します。

In [ ]:
df = pd.read_csv("./custom_mcp_agent/sku_data.csv")
df


### バックグラウンド MCP サーバーのクリーンアップ

`adk web` を実行した後に Jupyter カーネルを中断した時点でローカル MCP サーバーは終了しているはずですが、デプロイに進む前に、ポート `4200` をリッスンしている残存するバックグラウンドの `mcp_server` プロセスをクリーンに終了させるため、下のセルを実行します。

In [ ]:
if 'server_process' in locals() and server_process.poll() is None:
    print("Terminating Inventory MCP Server...")
    server_process.terminate()
    server_process.wait()
    print("SUCCESS: Inventory MCP Server terminated.")
else:
    print("MCP Server was not running or already terminated.")


## MCP サーバーの Google Cloud Run へのデプロイ

エンタープライズや本番環境では、MCP サーバーをローカルのバックグラウンドサブプロセスとして実行することはありません。代わりに、MCP サーバーを **Google Cloud Run** 上のスタンドアロンマイクロサービスとしてデプロイすることで、組織内の承認された任意の AI エージェントが HTTPS 経由でその `/sse` エンドポイントに安全に接続できるようにします。

このセクションでは、以下を行います。
1. コンテナ設定ファイル（`requirements.txt` および `Dockerfile`）を作成します。
2. `gcloud run deploy` を使用して、在庫 MCP サーバーを Google Cloud Run にデプロイします。
3. デプロイされた Cloud Run サービスの URL を取得し、リモート ADK エージェントをステップバイステップで接続します。

### Cloud Run 設定ファイルの作成

まず、MCP サーバーに必要な SDK の依存関係（`mcp>=1.25.0,<2.0.0`、`pydantic>=2.0.0`、`uvicorn>=0.30.0`）を記述した `requirements.txt` ファイルと、`mcp_server.py` および `sku_data.csv` をコンテナイメージにパッケージ化する `Dockerfile` を作成します。`FastMCP` との互換性を確保するため、`mcp` は `<2.0.0` に制約していることに注意してください。

Cloud Run は自動的に `PORT` 環境変数（デフォルトは `8080`）を注入し、`FastMCP` サーバーは `int(os.environ.get("PORT", 4200))` を介してこれを読み取ります。

In [ ]:
%%writefile ./custom_mcp_agent/requirements.txt
mcp>=1.25.0,<2.0.0
pydantic>=2.0.0
uvicorn>=0.30.0


In [ ]:
%%writefile ./custom_mcp_agent/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app

ENV PORT=8080
EXPOSE 8080

CMD ["python", "mcp_server.py"]


### Cloud Run CLI を使用したサービスのデプロイ

`--source` フラグを付けて Google Cloud SDK（`gcloud run deploy`）を使用します。Cloud Run は `Dockerfile` からコンテナイメージを自動的にビルドし、Google Cloud プロジェクトにサービスをデプロイします。

**注:** Cloud Run へのデプロイには通常 2〜3 分程度かかります。デプロイが完了すると、`gcloud` は安全な HTTPS サービス URL（例: `https://inventory-mcp-server-xxxxxx-uc.a.run.app`）を出力します。

In [ ]:
%%bash
gcloud run deploy inventory-mcp-server \
    --source=./custom_mcp_agent \
    --region=$CLOUD_RUN_LOCATION \
    --project=$GOOGLE_CLOUD_PROJECT \
    --allow-unauthenticated \
    --quiet


### デプロイされた Cloud Run サービス URL の取得

デプロイが完了したら、`gcloud run services describe` CLI コマンドを使用してデプロイされた Cloud Run サービスのアクティブな HTTPS URL を取得し、末尾に `/sse` を追加して完全な Model Context Protocol SSE エンドポイント URL を形成します。これを `REMOTE_MCP_SERVER_URL` 環境変数に保存します。

In [ ]:
# デプロイされた Cloud Run サービスの URL を取得
import os

SERVICE_URL = !gcloud run services describe "inventory-mcp-server" \
  --platform managed \
  --region $CLOUD_RUN_LOCATION \
  --project $PROJECT_ID \
  --format "value(status.url)"

remote_sse_url = f"{SERVICE_URL[0]}/sse"
os.environ["REMOTE_MCP_SERVER_URL"] = remote_sse_url
print("SUCCESS: Remote MCP Server SSE URL:", remote_sse_url)

# REMOTE_MCP_SERVER_URL を custom_mcp_agent/.env ファイルに永続化
env_file = "./custom_mcp_agent/.env"
if os.path.exists(env_file):
    with open(env_file, "a") as f:
        f.write(f"\nREMOTE_MCP_SERVER_URL={remote_sse_url}\n")

### Cloud Run に接続するための ADK エージェントの更新（`agent.py`）

MCP サーバーが Google Cloud Run にデプロイされたので、ローカルホスト（`http://0.0.0.0:4200/sse`）の代わりに HTTPS 経由でリモートの Cloud Run `/sse` エンドポイントに接続するように `./custom_mcp_agent/agent/agent.py` を更新します。

`MCP_SERVER_URL` が環境から `REMOTE_MCP_SERVER_URL` を読み取り、Cloud Run サービス URL にフォールバックする仕組みに注目してください。

In [ ]:
%%writefile ./custom_mcp_agent/agent/agent.py
"""MCP Toolset を使用して在庫管理ツールを作成する例。"""

import os

from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

# .env ファイルから環境変数を読み込む
env_path = os.path.abspath(
    os.path.join(os.path.dirname(__file__), "..", ".env")
)
if os.path.exists(env_path):
    load_dotenv(env_path, override=True)
else:
    load_dotenv(override=True)

MODEL = "gemini-3.5-flash"

# HTTPS 経由でリモートの Cloud Run MCP サーバーに接続
MCP_SERVER_URL = os.environ.get(
    "REMOTE_MCP_SERVER_URL",
    "https://inventory-mcp-server-xxxxxxxx-uc.a.run.app/sse",
)

root_agent = LlmAgent(
    model=Gemini(
        model=MODEL,
        client_kwargs={"vertexai": True, "location": "global"},
    ),
    name="inventory_assistant",
    description="You are a specialized assistant for inventory management.",
    instruction=(
        "Help user get answer to their queries about inventory and update "
        "or process the items."
    ),
    tools=[
        McpToolset(
            connection_params=SseServerParams(url=MCP_SERVER_URL),
        )
    ],
)

### リモート Cloud Run MCP エージェントのテスト

ADK の `Runner` を使用して更新された `inventory_assistant` エージェントをプログラムからテストし、エージェントが HTTPS 経由で Google Cloud Run に正常に接続し、リモートの `list_skus` ツールを呼び出すことを確認しましょう。

In [ ]:
from custom_mcp_agent.agent import agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
import importlib

importlib.reload(agent)

USER_ID = "user_1"
session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name="remote_mcp_test", user_id=USER_ID, session_id="remote_s1"
)

remote_runner = Runner(
    agent=agent.root_agent,
    app_name="remote_mcp_test",
    session_service=session_service,
)

query = "Can you list all available SKUs along with their product names, cost, and quantities on hand?"
print(f"\n=======================================================")
print(f">>> User Query: {query}")
print(f"=======================================================")

content = types.Content(role="user", parts=[types.Part(text=query)])
final_response_text = "Agent did not produce a final response."

async for event in remote_runner.run_async(
    user_id=USER_ID, session_id="remote_s1", new_message=content
):
    if event.is_final_response():
        if event.content and event.content.parts:
            final_response_text = event.content.parts[0].text
        break

print(f"<<< Agent Response:\n{final_response_text}\n")


### ADK Developer UI でのリモートエージェントの検査

グラフィカルな Developer UI（`adk web`）を使用して、リモートの Cloud Run MCP エージェントをテストおよび検査することもできます。

1. 下のセルを実行して ADK Developer UI を起動します。
2. 出力された URL を開き、上部のエージェントドロップダウンから **`agent`** を選択します（これでリモートの Cloud Run MCP サーバーを利用するエージェントになります）。
3. サンプル質問を送信して Cloud Run MCP サービスをテストします。
   * *"What is the current stock level and status of SKU001 (Laptop Pro 15)?"*（SKU001（Laptop Pro 15）の現在の在庫レベルとステータスはどうなっていますか？）
   * *"Can you list all available SKUs along with their quantities on hand?"*（利用可能なすべての SKU とその手持ち数量を一覧表示できますか？）
   * *"We just received 5 more units of SKU001. Please add 5 to the quantity of SKU001."*（SKU001 がさらに 5 台入荷しました。SKU001 の数量に 5 を追加してください。）
4. **Events** パネルでリモート HTTPS SSE ツールの呼び出しを検査します。
5. **Cloud Run のメトリクスの確認**: リモートエージェントとやり取りした後、[Cloud Run の可観測性ページ](https://console.cloud.google.com/run/detail/us-central1/inventory-mcp-server/observability/metrics) を確認して、エージェントのツール呼び出しによって生成された受信リクエスト数やコンテナメトリクスを観察します。
6. 確認が終わったら、Jupyter で **`Interrupt the kernel`**（カーネルを中断）をクリックします。

In [ ]:
# Cloud Workstations の場合
!adk web custom_mcp_agent --allow_origins "regex:https://.*\.cloudworkstations\.dev" 

**注:** Agent Platform Workbench を使用している場合は、コメントアウトを解除して以下のセルを実行してください。

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web custom_mcp_agent --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"


#### Google Cloud Console での Cloud Run メトリクスの確認

Developer UI でリモート MCP エージェントとやり取りした後、Google Cloud Console を確認して、エージェントのツール呼び出しがリモートサーバー上で呼び出されたことを検証できます。

* `inventory-mcp-server` の **[Cloud Run の可観測性とメトリクス ページ](https://console.cloud.google.com/run/detail/us-central1/inventory-mcp-server/observability/metrics)** を開きます。
* Developer UI からの SSE ツール呼び出しに対応して、**Request count（リクエスト数）**、**Container CPU utilization（コンテナ CPU 使用率）**、**Request latency（リクエスト レイテンシ）** が急増していることを確認してください。

---

### 本番環境における考慮事項: サービスアカウントと IAM による Cloud Run の保護

利便性のため、ここでは Cloud Run サービスを `--allow-unauthenticated` でデプロイしました。本番環境では、内部 MCP サーバーを一般に公開することは**決して避けるべき**です。

本番環境で Cloud Run MCP サービスを保護するには:
1. **`--allow-unauthenticated` を省略する** 
2. エージェントのサービスアカウントに **Cloud Run 起動元（`roles/run.invoker`）** を付与する:
   ```bash
   gcloud run services add-iam-policy-binding inventory-mcp-server \
       --region=$CLOUD_RUN_LOCATION \
       --project=$GOOGLE_CLOUD_PROJECT \
       --member="serviceAccount:YOUR_AGENT_SA@$GOOGLE_CLOUD_PROJECT.iam.gserviceaccount.com" \
       --role="roles/run.invoker"
   ```
3. **`agent.py` で OIDC ID トークンヘッダーを渡す**:
   Google ADK の `SseServerParams` はネイティブに HTTP ヘッダーを受け入れます。Cloud Run の Audience をターゲットにした署名付き OpenID Connect（OIDC）ID トークンを取得し、`Authorization: Bearer <ID_TOKEN>` ヘッダーを渡すことができます。
   ```python
   from google.auth.transport.requests import Request
   from google.oauth2 import id_token
   from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
   from google.adk.tools.mcp_tool.mcp_toolset import McpToolset

   # 1. ID トークンのベース Audience（/sse パスを除いた Cloud Run のルート URL）
   audience = MCP_SERVER_URL.replace("/sse", "").rstrip("/")

   # 2. サービスアカウント認証用の OIDC ID トークンを取得
   headers = {}
   try:
       auth_req = Request()
       token = id_token.fetch_id_token(auth_req, audience=audience)
       headers["Authorization"] = f"Bearer {token}"
   except Exception:
       pass

   # 3. 認証ヘッダーを SseServerParams に渡す
   mcp_toolset = McpToolset(
       connection_params=SseServerParams(
           url=MCP_SERVER_URL,
           headers=headers,
       )
   )
   ```

## まとめ

お疲れ様でした。**Model Context Protocol（MCP）** と Google の **Agent Development Kit（ADK）** を使用して、Generative AI エージェントの構築、テスト、検査を無事に完了しました。

* **MCP サーバーの統合**: `FastMCP`（`@mcp.tool()`）を使用して、読み取り専用ツール（`list_skus`）と状態変更ツール（`update_sku_qty`）の両方を備えた在庫管理 MCP サーバーを構築しました。
* **ADK McpToolset**: `McpToolset` と `SseServerParams` を使用して、SSE 経由でローカルサーバーに ADK の `LlmAgent` を接続しました。
* **ADK Developer UI**: `adk web` を起動して、エージェントと MCP 間の通信を対話形式で検査およびテストし、リアルタイムの在庫更新を確認しました。
* **Cloud Run 本番デプロイ**: `Dockerfile` を使用して MCP サーバーをパッケージ化し、`gcloud run deploy` を使用してスタンドアロンのクラウドマイクロサービスとしてデプロイし、そのリモート HTTPS `/sse` エンドポイントに接続するように ADK エージェント（`agent.py`）を更新しました。

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.